In [ ]:

from pyspark.sql.functions import (
    col,
    date_format,
    to_date,
    year,
    month,
    dayofmonth,
    dayofweek,
    hour,
    minute,
    second,
    quarter,
    weekofyear,
    unix_timestamp,
    date_trunc,
    round
)

bronze_df = spark.table("workspace.bronze.nyctaxi_trips_raw")

silver_df = (
    bronze_df
    .select(
        # keep original timestamp columns
        col("tpep_pickup_datetime"),
        col("tpep_dropoff_datetime"),

        # business columns
        col("trip_distance"),
        col("fare_amount"),
        col("pickup_zip"),
        col("dropoff_zip"),

        # metadata columns
        col("ingested_at"),
        col("_source_table"),

        # pickup date formats
        to_date("tpep_pickup_datetime").alias("pickup_date"),
        date_format("tpep_pickup_datetime", "yyyy-MM-dd").alias("pickup_format_yyyy_mm_dd"),
        date_format("tpep_pickup_datetime", "dd-MM-yyyy").alias("pickup_format_dd_mm_yyyy"),
        date_format("tpep_pickup_datetime", "MM/dd/yyyy").alias("pickup_format_mm_dd_yyyy"),
        date_format("tpep_pickup_datetime", "yyyyMMdd").alias("pickup_format_yyyymmdd"),

        # pickup datetime formats
        date_format("tpep_pickup_datetime", "yyyy-MM-dd HH:mm:ss").alias("pickup_datetime_24h"),
        date_format("tpep_pickup_datetime", "yyyy-MM-dd hh:mm:ss a").alias("pickup_datetime_12h"),
        date_format("tpep_pickup_datetime", "dd MMM yyyy HH:mm").alias("pickup_datetime_short_month"),
        date_format("tpep_pickup_datetime", "dd MMMM yyyy HH:mm:ss").alias("pickup_datetime_full_month"),
        date_format("tpep_pickup_datetime", "EEEE, dd MMMM yyyy").alias("pickup_full_day_name"),

        # pickup time only
        date_format("tpep_pickup_datetime", "HH:mm:ss").alias("pickup_time_24h"),
        date_format("tpep_pickup_datetime", "hh:mm:ss a").alias("pickup_time_12h"),

        # pickup date parts
        year("tpep_pickup_datetime").alias("pickup_year"),
        quarter("tpep_pickup_datetime").alias("pickup_quarter"),
        month("tpep_pickup_datetime").alias("pickup_month"),
        dayofmonth("tpep_pickup_datetime").alias("pickup_day"),
        dayofweek("tpep_pickup_datetime").alias("pickup_day_of_week"),
        weekofyear("tpep_pickup_datetime").alias("pickup_week_of_year"),
        hour("tpep_pickup_datetime").alias("pickup_hour"),
        minute("tpep_pickup_datetime").alias("pickup_minute"),
        second("tpep_pickup_datetime").alias("pickup_second"),

        # unix timestamp
        unix_timestamp("tpep_pickup_datetime").alias("pickup_unix_timestamp"),

        # truncated timestamps
        date_trunc("DAY", col("tpep_pickup_datetime")).alias("pickup_trunc_day"),
        date_trunc("MONTH", col("tpep_pickup_datetime")).alias("pickup_trunc_month"),
        date_trunc("YEAR", col("tpep_pickup_datetime")).alias("pickup_trunc_year"),

        # dropoff examples
        to_date("tpep_dropoff_datetime").alias("dropoff_date"),
        date_format("tpep_dropoff_datetime", "yyyy-MM-dd HH:mm:ss").alias("dropoff_datetime_24h"),
        date_format("tpep_dropoff_datetime", "dd-MM-yyyy HH:mm:ss").alias("dropoff_datetime_eu"),
        date_format("tpep_dropoff_datetime", "MM/dd/yyyy hh:mm:ss a").alias("dropoff_datetime_us_12h"),

        # simple calculated column
        round(
            (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60,
            2
        ).alias("trip_duration_minutes")
    )
    .where(col("trip_distance") > 0)
    .where(col("fare_amount") >= 0)
)

silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.nyctaxi_trips_clean")

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip,ingested_at,_source_table,pickup_date,pickup_format_yyyy_mm_dd,pickup_format_dd_mm_yyyy,pickup_format_mm_dd_yyyy,pickup_format_yyyymmdd,pickup_datetime_24h,pickup_datetime_12h,pickup_datetime_short_month,pickup_datetime_full_month,pickup_full_day_name,pickup_time_24h,pickup_time_12h,pickup_year,pickup_quarter,pickup_month,pickup_day,pickup_day_of_week,pickup_week_of_year,pickup_hour,pickup_minute,pickup_second,pickup_unix_timestamp,pickup_trunc_day,pickup_trunc_month,pickup_trunc_year,dropoff_date,dropoff_datetime_24h,dropoff_datetime_eu,dropoff_datetime_us_12h,trip_duration_minutes
0,2016-02-13 21:47:53,2016-02-13 21:57:15,1.40,8.0,10103,10110,2026-05-07 11:06:46.094481,samples.nyctaxi.trips,2016-02-13,2016-02-13,13-02-2016,02/13/2016,20160213,2016-02-13 21:47:53,2016-02-13 09:47:53 PM,13 Feb 2016 21:47,13 February 2016 21:47:53,"Saturday, 13 February 2016",21:47:53,09:47:53 PM,2016,1,2,13,7,6,21,47,53,1455400073,2016-02-13,2016-02-01,2016-01-01,2016-02-13,2016-02-13 21:57:15,13-02-2016 21:57:15,02/13/2016 09:57:15 PM,9.37
1,2016-02-13 18:29:09,2016-02-13 18:37:23,1.31,7.5,10023,10023,2026-05-07 11:06:46.094481,samples.nyctaxi.trips,2016-02-13,2016-02-13,13-02-2016,02/13/2016,20160213,2016-02-13 18:29:09,2016-02-13 06:29:09 PM,13 Feb 2016 18:29,13 February 2016 18:29:09,"Saturday, 13 February 2016",18:29:09,06:29:09 PM,2016,1,2,13,7,6,18,29,9,1455388149,2016-02-13,2016-02-01,2016-01-01,2016-02-13,2016-02-13 18:37:23,13-02-2016 18:37:23,02/13/2016 06:37:23 PM,8.23
2,2016-02-06 19:40:58,2016-02-06 19:52:32,1.80,9.5,10001,10018,2026-05-07 11:06:46.094481,samples.nyctaxi.trips,2016-02-06,2016-02-06,06-02-2016,02/06/2016,20160206,2016-02-06 19:40:58,2016-02-06 07:40:58 PM,06 Feb 2016 19:40,06 February 2016 19:40:58,"Saturday, 06 February 2016",19:40:58,07:40:58 PM,2016,1,2,6,7,5,19,40,58,1454787658,2016-02-06,2016-02-01,2016-01-01,2016-02-06,2016-02-06 19:52:32,06-02-2016 19:52:32,02/06/2016 07:52:32 PM,11.57
3,2016-02-12 19:06:43,2016-02-12 19:20:54,2.30,11.5,10044,10111,2026-05-07 11:06:46.094481,samples.nyctaxi.trips,2016-02-12,2016-02-12,12-02-2016,02/12/2016,20160212,2016-02-12 19:06:43,2016-02-12 07:06:43 PM,12 Feb 2016 19:06,12 February 2016 19:06:43,"Friday, 12 February 2016",19:06:43,07:06:43 PM,2016,1,2,12,6,6,19,6,43,1455304003,2016-02-12,2016-02-01,2016-01-01,2016-02-12,2016-02-12 19:20:54,12-02-2016 19:20:54,02/12/2016 07:20:54 PM,14.18
4,2016-02-23 10:27:56,2016-02-23 10:58:33,2.60,18.5,10199,10022,2026-05-07 11:06:46.094481,samples.nyctaxi.trips,2016-02-23,2016-02-23,23-02-2016,02/23/2016,20160223,2016-02-23 10:27:56,2016-02-23 10:27:56 AM,23 Feb 2016 10:27,23 February 2016 10:27:56,"Tuesday, 23 February 2016",10:27:56,10:27:56 AM,2016,1,2,23,3,8,10,27,56,1456223276,2016-02-23,2016-02-01,2016-01-01,2016-02-23,2016-02-23 10:58:33,23-02-2016 10:58:33,02/23/2016 10:58:33 AM,30.62
5,2016-02-13 00:41:43,2016-02-13 00:46:52,1.40,6.5,10023,10069,2026-05-07 11:06:46.094481,samples.nyctaxi.trips,2016-02-13,2016-02-13,13-02-2016,02/13/2016,20160213,2016-02-13 00:41:43,2016-02-13 12:41:43 AM,13 Feb 2016 00:41,13 February 2016 00:41:43,"Saturday, 13 February 2016",00:41:43,12:41:43 AM,2016,1,2,13,7,6,0,41,43,1455324103,2016-02-13,2016-02-01,2016-01-01,2016-02-13,2016-02-13 00:46:52,13-02-2016 00:46:52,02/13/2016 12:46:52 AM,5.15
6,2016-02-18 23:49:53,2016-02-19 00:12:53,10.40,31.0,11371,10003,2026-05-07 11:06:46.094481,samples.nyctaxi.trips,2016-02-18,2016-02-18,18-02-2016,02/18/2016,20160218,2016-02-18 23:49:53,2016-02-18 11:49:53 PM,18 Feb 2016 23:49,18 February 2016 23:49:53,"Thursday, 18 February 2016",23:49:53,11:49:53 PM,2016,1,2,18,5,7,23,49,53,1455839393,2016-02-18,2016-02-01,2016-01-01,2016-02-19,2016-02-19 00:12:53,19-02-2016 00:12:53,02/19/2016 12:12:53 AM,23.00
7,2016-02-18 20:21:45,2016-02-18 20:38:23,10.15,28.5,11371,11201,2026-05-07 11:06:46.094481,samples.nyctaxi.trips,2016-02-18,2016-02-18,18-02-2016,02/18/2016,20160218